# Production-Level RAG Evaluation

This notebook demonstrates a real-world pipeline:
1. Load your actual resume.
2. Chunk and index it in a Vector DB.
3. Load a golden evaluation dataset from a JSON file.
4. Run the pipeline and evaluate it with LLM-as-a-Judge.

In [ ]:
import re
import numpy as np
from typing import List

from mutant.providers import OllamaProvider, LLMMessage
from mutant.eval import evaluate
from mutant.datasets import load_test_cases
from mutant.eval.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

# 1. Initialize the Local Model
provider = OllamaProvider(model="llama3.2")


## 1. Build the Vector Database
We'll use a simple in-memory TF-IDF VectorStore to index your resume chunks.

In [ ]:
class TFIDFVectorStore:
    def __init__(self):
        self.documents = []
        self.vocab = {}
        self.idf = None
        self.tf_idf_matrix = None
        
    def _tokenize(self, text: str) -> List[str]:
        return re.findall(r'\b\w+\b', text.lower())
        
    async def add_documents(self, docs: List[str]):
        print(f"Indexing {len(docs)} chunks of your Resume...")
        self.documents.extend(docs)
        
        doc_words = [self._tokenize(d) for d in docs]
        words = set([w for doc in doc_words for w in doc])
        self.vocab = {w: i for i, w in enumerate(words)}
        
        tf = np.zeros((len(docs), len(self.vocab)))
        for i, doc in enumerate(doc_words):
            for w in doc:
                tf[i, self.vocab[w]] += 1
                
        df = np.sum(tf > 0, axis=0)
        self.idf = np.log(len(docs) / (df + 1))
        
        self.tf_idf_matrix = tf * self.idf
        norms = np.linalg.norm(self.tf_idf_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1
        self.tf_idf_matrix = self.tf_idf_matrix / norms
        print("✅ Vector DB Ready!")

    async def search(self, query: str, top_k: int = 3) -> List[str]:
        q_words = self._tokenize(query)
        q_vec = np.zeros(len(self.vocab))
        for w in q_words:
            if w in self.vocab:
                q_vec[self.vocab[w]] += 1
                
        q_vec = q_vec * self.idf
        norm = np.linalg.norm(q_vec)
        if norm > 0:
            q_vec = q_vec / norm
            
        similarities = np.dot(self.tf_idf_matrix, q_vec)
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [self.documents[i] for i in top_indices]


## 2. Load the Resume
We'll chunk `resume_text.txt` and add it to the Vector DB.

In [ ]:
with open("../../resume_text.txt", "r") as f:
    resume_text = f.read()

# Chunk by paragraphs
raw_chunks = [c.strip() for c in resume_text.split("\n\n") if len(c.strip()) > 50]

vector_db = TFIDFVectorStore()
await vector_db.add_documents(raw_chunks)


## 3. The RAG Pipeline Agent
This agent queries the DB, stores the retrieval context, and answers the question.

In [ ]:
class ResumeAgent:
    def __init__(self, db, llm_provider):
        self.db = db
        self.llm = llm_provider

    async def answer_and_record(self, test_case):
        # Retrieve chunks
        retrieved_docs = await self.db.search(test_case.input, top_k=3)
        
        # Log retrieval context directly into the TestCase!
        test_case.retrieval_context = retrieved_docs
        
        # Generate Answer
        context_str = "\n\n---\n\n".join(retrieved_docs)
        prompt = f"You are an AI assistant for Ankit Mishra. Answer strictly using this context:\n{context_str}\n\nQuestion: {test_case.input}"
        
        messages = [LLMMessage(role="user", content=prompt)]
        response = await self.llm.complete(messages, temperature=0.2)
        
        # Log actual output
        test_case.actual_output = response.content
        return test_case

agent = ResumeAgent(vector_db, provider)


## 4. Load Dataset & Evaluate
Now we load the `resume_eval_data.json` dataset directly from disk, run it through the RAG pipeline, and evaluate!

In [ ]:
print("📂 Loading Golden Dataset from JSON...")
dataset = load_test_cases("resume_eval_data.json")

print(f"🚀 Running {len(dataset)} cases through the RAG Pipeline...")
for tc in dataset:
    await agent.answer_and_record(tc)

print("⚖️ Evaluating Pipeline (Faithfulness, Relevancy, Precision, Recall)...")
report = await evaluate(
    dataset,
    metrics=[
        Faithfulness(provider),
        AnswerRelevancy(provider),
        ContextPrecision(provider),
        ContextRecall(provider)
    ],
    concurrency=1  # Prevent local GPU from overloading
)

# Render HTML Dashboard
report.display(html_path="production_rag_dashboard.html")
